## Setup

In [2]:
%load_ext autoreload
%autoreload 2

import torch
import ww_classes as ww

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 01 Vocabulary and Dataset

In [2]:
ww.corpus

['We always come to Paris',
 'The professor is from Australia',
 'I live in Stanford',
 'He comes from Taiwan',
 'The capital of Turkey is Ankara']

In [3]:
train_dataset = ww.WWDataset(ww.corpus)

In [4]:
len(train_dataset)

5

In [7]:
for data in train_dataset:
    print(data)

(['we', 'always', 'come', 'to', 'paris'], [0, 0, 0, 0, 1])
(['the', 'professor', 'is', 'from', 'australia'], [0, 0, 0, 0, 1])
(['i', 'live', 'in', 'stanford'], [0, 0, 0, 1])
(['he', 'comes', 'from', 'taiwan'], [0, 0, 0, 1])
(['the', 'capital', 'of', 'turkey', 'is', 'ankara'], [0, 0, 0, 1, 0, 1])


In [8]:
vocab = ww.WWVocabulary(train_dataset.data)

In [9]:
len(vocab)

23

In [11]:
list(vocab.vocab)[:5]

['<pad>', '<unk>', 'always', 'ankara', 'australia']

In [12]:
list(vocab.idx2word.items())[:5]

[(0, '<pad>'), (1, '<unk>'), (2, 'always'), (3, 'ankara'), (4, 'australia')]

## 02 Dataloader

In [ ]:
train_dataset = ww.WWDataset(ww.corpus)
batch = [train_dataset[i] for i in range(2)]

In [15]:
batch

[(['we', 'always', 'come', 'to', 'paris'], [0, 0, 0, 0, 1]),
 (['the', 'professor', 'is', 'from', 'australia'], [0, 0, 0, 0, 1])]

### DEBUGGING `zip(*batch)`

In [20]:
list(zip(*batch))

[(['we', 'always', 'come', 'to', 'paris'],
  ['the', 'professor', 'is', 'from', 'australia']),
 ([0, 0, 0, 0, 1], [0, 0, 0, 0, 1])]

In [24]:
a1 = ('x1', 'y1')
a2 = ('x2', 'y2')
a = [a1, a2]

In [25]:
list(zip(*a))

[('x1', 'x2'), ('y1', 'y2')]

In [26]:
list(zip(a1, a2))

[('x1', 'x2'), ('y1', 'y2')]

In [16]:
x, y = zip(*batch)
x, y

((['we', 'always', 'come', 'to', 'paris'],
  ['the', 'professor', 'is', 'from', 'australia']),
 ([0, 0, 0, 0, 1], [0, 0, 0, 0, 1]))

### DEBUGGING `WWCollate`

In [7]:
config = ww.WWConfig()
train_dataset = ww.WWDataset(ww.corpus)
vocab = ww.WWVocabulary(train_dataset.data)
batch = [train_dataset[i] for i in range(2)]

In [8]:
batch

[(['we', 'always', 'come', 'to', 'paris'], [0, 0, 0, 0, 1]),
 (['the', 'professor', 'is', 'from', 'australia'], [0, 0, 0, 0, 1])]

In [20]:
sentences, labels = zip(*batch)
sentences

(['we', 'always', 'come', 'to', 'paris'],
 ['the', 'professor', 'is', 'from', 'australia'])

In [16]:
def pad_window(sentence, window_size, vocab):
        window = [vocab.PAD_TOKEN] * window_size
        return window + sentence + window

In [21]:
sentences = [pad_window(sent, config.window_size, vocab) for sent in sentences]
sentences

[['<pad>', '<pad>', 'we', 'always', 'come', 'to', 'paris', '<pad>', '<pad>'],
 ['<pad>',
  '<pad>',
  'the',
  'professor',
  'is',
  'from',
  'australia',
  '<pad>',
  '<pad>']]

In [22]:
vocab.encode_batch(sentences)

[[0, 0, 22, 2, 6, 20, 15, 0, 0], [0, 0, 19, 16, 12, 8, 4, 0, 0]]

## 03 Word Window Classifier

### intro

In [3]:
config = ww.WWConfig()
train_dataset = ww.WWDataset(ww.corpus)
vocab = ww.WWVocabulary(train_dataset.data)
loader = ww.get_data_loader(ww.corpus, train_dataset, config, vocab)

In [4]:
batch = next(iter(loader))

In [5]:
batch

(tensor([[ 0,  0, 22,  2,  6, 20, 15,  0,  0],
         [ 0,  0, 10, 13, 11, 17,  0,  0,  0]]),
 tensor([[0., 0., 0., 0., 1.],
         [0., 0., 0., 1., 0.]]),
 tensor([5, 4]))

In [6]:
batch_inputs, batch_labels, batch_lengths = batch

### `__init__` Method 

In [7]:
# [B, L]
batch_inputs.size()

torch.Size([2, 9])

In [8]:
batch_inputs

tensor([[ 0,  0, 22,  2,  6, 20, 15,  0,  0],
        [ 0,  0, 10, 13, 11, 17,  0,  0,  0]])

In [9]:
token_windows = batch_inputs.unfold(1, 2 * config.window_size + 1, 1)

In [10]:
token_windows

tensor([[[ 0,  0, 22,  2,  6],
         [ 0, 22,  2,  6, 20],
         [22,  2,  6, 20, 15],
         [ 2,  6, 20, 15,  0],
         [ 6, 20, 15,  0,  0]],

        [[ 0,  0, 10, 13, 11],
         [ 0, 10, 13, 11, 17],
         [10, 13, 11, 17,  0],
         [13, 11, 17,  0,  0],
         [11, 17,  0,  0,  0]]])

In [11]:
x = torch.tensor([10, 20, 30, 40, 50, 60, 70])
# We want windows of size 3, sliding by 1 step:
windows = x.unfold(dimension=0, size=3, step=1)

In [12]:
windows

tensor([[10, 20, 30],
        [20, 30, 40],
        [30, 40, 50],
        [40, 50, 60],
        [50, 60, 70]])

### `forward` Method 

In [ ]:
model = ww.WWClassifierBin(config=config, 
                           vocab_size=len(vocab), 
                           pad_ix=vocab.token_pad_idx, 
                           loss_type='bce_with_logits')

In [18]:
model(batch_inputs);

Logits:
tensor([[-0.1431,  0.4615,  0.1612,  0.0879, -0.0353],
        [ 0.0758,  0.2884,  0.5992,  0.0402,  0.1388]],
       grad_fn=<SqueezeBackward1>)
Output:
tensor([[-0.1431,  0.4615,  0.1612,  0.0879, -0.0353],
        [ 0.0758,  0.2884,  0.5992,  0.0402,  0.1388]],
       grad_fn=<ViewBackward0>)


In [ ]:
model_bce = ww.WWClassifierBin(config=config, 
                           vocab_size=len(vocab), 
                           pad_ix=vocab.token_pad_idx, 
                           loss_type='bce')

In [20]:
model_bce(batch_inputs);

Logits:
tensor([[-0.2216,  0.1858,  0.2568,  0.0313, -0.0313],
        [-0.2183, -0.2114, -0.1938,  0.1683,  0.2539]],
       grad_fn=<SqueezeBackward1>)

Output:
tensor([[0.4448, 0.5463, 0.5638, 0.5078, 0.4922],
        [0.4456, 0.4473, 0.4517, 0.5420, 0.5631]], grad_fn=<ViewBackward0>)


In [22]:
torch.sigmoid(torch.tensor(-0.2216))

tensor(0.4448)

### Debugging padding in collate function

In [44]:
torch.manual_seed(42)
config = ww.WWConfig()
train_dataset = ww.WWDataset(ww.corpus)
vocab = ww.WWVocabulary(train_dataset.data)
loader = ww.get_data_loader(ww.corpus, train_dataset, config, vocab)
model = ww.WWClassifierBin(config=config, vocab=vocab)

In [45]:
batch = next(iter(loader))
batch_inputs, batch_labels, batch_lengths = batch
batch_inputs

tensor([[ 0,  0, 10, 13, 11, 17,  0,  0,  0],
        [ 0,  0, 19, 16, 12,  8,  4,  0,  0]])

In [27]:
model(batch_inputs);

Logits:
tensor([[ 0.4051,  0.0383,  0.0578,  0.4674,  0.4075],
        [ 0.2192,  0.3621,  0.2759,  0.1301, -0.2752]],
       grad_fn=<SqueezeBackward1>)

Output:
tensor([[ 0.4051,  0.0383,  0.0578,  0.4674,  0.4075],
        [ 0.2192,  0.3621,  0.2759,  0.1301, -0.2752]],
       grad_fn=<ViewBackward0>)


## 04 Training

In [49]:
torch.manual_seed(42)
onfig = ww.WWConfig()
train_dataset = ww.WWDataset(ww.corpus)
vocab = ww.WWVocabulary(train_dataset.data)

In [50]:
trainer = ww.TrainerBin(config=config, 
                        corpus=ww.corpus, 
                        dataset=train_dataset, 
                        vocab=vocab,
                        loss_type='bce_with_logits')

In [51]:
trainer.train()

Epoch 0, Loss: 0.3363
Epoch 100, Loss: 0.2560
Epoch 200, Loss: 0.1963
Epoch 300, Loss: 0.1458
Epoch 400, Loss: 0.1104
Epoch 500, Loss: 0.0810
Epoch 600, Loss: 0.0627
Epoch 700, Loss: 0.0583
Epoch 800, Loss: 0.0445
Epoch 900, Loss: 0.0330
